# Cosine Similarity: From Triangles to Dot Products

This notebook connects two ideas that seem unrelated:
1. **Triangle definition**: cos = adjacent / hypotenuse (needs a 90-degree angle)
2. **Dot product formula**: cos = A.B / (|A| * |B|) (no triangle in sight)

They're actually the **same thing**. The dot product secretly builds a right triangle.

We'll prove it step by step.

---
## Part 1: The Triangle Definition (from school)

In a right triangle, cosine is defined as:

```
cos(angle) = adjacent / hypotenuse
```

This **requires** a 90-degree angle:

```
         /|
        / |
  hyp  /  |  opposite
      /   |
     / a  |
    /_____|         <-- 90 degrees here
    adjacent
```

Some familiar values:

| Angle | cos | Meaning |
|-------|-----|-----|
| 0     | 1.0 | adjacent = hypotenuse (flat line) |
| 45    | 0.707 | adjacent = hyp / sqrt(2) |
| 60    | 0.5 | adjacent = half of hyp |
| 90    | 0.0 | adjacent = 0 (straight up) |

In [ ]:
import math

print("Triangle cosine values:")
print(f"{'Angle':>7} {'cos':>8} {'adjacent/hyp':>14}")
print("-" * 32)

for angle_deg in [0, 30, 45, 60, 90, 120, 180]:
    angle_rad = math.radians(angle_deg)
    cos_val = math.cos(angle_rad)
    
    # Show as fraction of hypotenuse
    hyp = 1.0
    adj = cos_val * hyp
    print(f"{angle_deg:>5}deg {cos_val:>+8.4f} {adj:>+6.3f} / {hyp:.1f}")

print("\nKey insight: cos(angle) tells you how much of the")
print("hypotenuse lies along the adjacent side.")
print("cos(0) = 100% along, cos(90) = 0% along.")

In [ ]:
# Visualize the right triangle for different angles

def draw_right_triangle(angle_deg):
    angle_rad = math.radians(angle_deg)
    cos_val = math.cos(angle_rad)
    sin_val = math.sin(angle_rad)
    
    # Grid
    W, H = 35, 12
    grid = [[' ' for _ in range(W)] for _ in range(H)]
    
    # Origin at bottom-left
    ox, oy = 2, H - 2
    
    # Scale
    sc = 8
    hyp_len = sc
    adj_len = cos_val * sc
    opp_len = sin_val * sc
    
    # Draw adjacent (horizontal)
    adj_px = int(abs(adj_len) * 3)
    for x in range(min(adj_px + 1, W - ox)):
        if ox + x < W:
            grid[oy][ox + x] = '-'
    
    # Draw opposite (vertical) at right end
    right_x = ox + adj_px
    opp_px = int(abs(opp_len) * 1)
    if right_x < W:
        for y in range(min(opp_px + 1, oy)):
            if oy - y >= 0:
                grid[oy - y][right_x] = '|'
    
    # Draw hypotenuse
    steps = max(adj_px, opp_px, 10)
    for t in range(steps + 1):
        frac = t / steps
        hx = ox + int(frac * adj_px)
        hy = oy - int(frac * opp_px)
        if 0 <= hx < W and 0 <= hy < H:
            if grid[hy][hx] == ' ':
                grid[hy][hx] = '/'
    
    # Mark 90-degree corner
    if right_x < W and oy < H:
        grid[oy][right_x] = '+'
    
    # Mark angle
    grid[oy][ox] = 'a'
    
    print(f"  Angle = {angle_deg}deg    cos({angle_deg}) = {cos_val:.3f}")
    print(f"  adjacent = {cos_val:.3f}    hypotenuse = 1.000")
    print()
    for row in grid:
        print('  ' + ''.join(row))
    print()

for angle in [30, 45, 60]:
    draw_right_triangle(angle)
    print()

---
## Part 2: Two Vectors and the Angle Between Them

Now forget triangles for a moment. We have two vectors:

```
A = [3, 4]
B = [4, 0]
```

They point in different directions. There's an **angle** between them.
But where's the right triangle? There isn't one — yet.

To use the triangle definition, we need to **create** a right angle.
We do this by **projecting** one vector onto the other.

In [ ]:
# Two vectors — just the raw setup

A = [3, 4]
B = [4, 0]

print("Two vectors:")
print(f"  A = {A}  (length = {math.sqrt(A[0]**2 + A[1]**2):.2f})")
print(f"  B = {B}  (length = {math.sqrt(B[0]**2 + B[1]**2):.2f})")
print()

# Draw them
W, H = 30, 12
grid = [[' ' for _ in range(W)] for _ in range(H)]

ox, oy = 2, H - 2

# Draw A
steps = 20
for t in range(steps + 1):
    frac = t / steps
    x = ox + int(frac * A[0] * 2)
    y = oy - int(frac * A[1] * 1.5)
    if 0 <= x < W and 0 <= y < H:
        grid[y][x] = 'A'

# Draw B
for t in range(steps + 1):
    frac = t / steps
    x = ox + int(frac * B[0] * 2)
    y = oy - int(frac * B[1] * 1.5)
    if 0 <= x < W and 0 <= y < H:
        grid[y][x] = 'B'

grid[oy][ox] = 'O'

for row in grid:
    print('  ' + ''.join(row))

print()
print("  There's an angle between A and B.")
print("  But no right triangle. How do we find cos(angle)?")
print("  Answer: PROJECTION creates the right triangle.")

---
## Part 3: Projection Creates the Right Triangle

**Projection** = drop a perpendicular from the tip of A onto the line of B.

This creates a right angle, which gives us a right triangle:

```
                  A = [3, 4]
                 /|
                / |
     |A|=5     /  |  (perpendicular — the RIGHT ANGLE)
              /   |
             / a  |
    O-------P-----+
    |  proj  |
    |  = 3   |
    
    B direction -->  [4, 0]
```

Now we have:
- **Hypotenuse** = |A| = 5
- **Adjacent** = projection of A onto B = 3
- **cos(angle) = adjacent / hypotenuse = 3 / 5 = 0.6**

The key question: **how do we compute the projection length?**

In [ ]:
# Show the projection creating a right triangle

A = [3, 4]
B = [4, 0]  # points along x-axis for simplicity

len_A = math.sqrt(A[0]**2 + A[1]**2)
len_B = math.sqrt(B[0]**2 + B[1]**2)

# Projection of A onto B direction
# proj = (A.B / |B|^2) * B, but the LENGTH is A.B / |B|
dot_AB = A[0]*B[0] + A[1]*B[1]
proj_length = dot_AB / len_B

print("Creating the right triangle by PROJECTION:")
print("=" * 50)
print()
print(f"  A = {A}     |A| = {len_A:.2f}  (this becomes the hypotenuse)")
print(f"  B = {B}     |B| = {len_B:.2f}  (this is the direction we project onto)")
print()

# Draw the triangle
W, H = 35, 14
grid = [[' ' for _ in range(W)] for _ in range(H)]

ox, oy = 3, H - 3
sc = 2.5

# Draw B direction (horizontal line)
for x in range(int(B[0] * sc) + 2):
    if ox + x < W:
        grid[oy][ox + x] = '-'

# Draw A vector
for t in range(20):
    frac = t / 19
    x = ox + int(frac * A[0] * sc)
    y = oy - int(frac * A[1] * sc * 0.7)
    if 0 <= x < W and 0 <= y < H and grid[y][x] == ' ':
        grid[y][x] = '/'

# Draw vertical projection line (the right angle)
proj_x = ox + int(A[0] * sc)
proj_top_y = oy - int(A[1] * sc * 0.7)
if proj_x < W:
    for y in range(proj_top_y, oy + 1):
        if 0 <= y < H:
            grid[y][proj_x] = '|'

# Labels
grid[oy][ox] = 'O'
tip_x = ox + int(A[0] * sc)
tip_y = oy - int(A[1] * sc * 0.7)
if 0 <= tip_x < W and 0 <= tip_y < H:
    grid[tip_y][tip_x] = 'A'
if proj_x < W:
    grid[oy][proj_x] = 'P'

for row in grid:
    print('  ' + ''.join(row))

print()
print(f"  O = origin")
print(f"  A = tip of vector A")
print(f"  P = projection point (where the perpendicular meets B)")
print(f"  | = perpendicular line (90-degree angle at P)")
print()
print(f"  Right triangle:")
print(f"    hypotenuse (O to A) = |A| = {len_A:.2f}")
print(f"    adjacent   (O to P) = projection = {proj_length:.2f}")
print(f"    cos(angle) = adjacent / hypotenuse = {proj_length:.2f} / {len_A:.2f} = {proj_length / len_A:.4f}")

---
## Part 4: Why the Dot Product Computes Projection

This is the key connection. The dot product has a **geometric meaning**:

```
A . B = |A| * |B| * cos(angle)
```

This isn't obvious from the formula `a1*b1 + a2*b2`. Let's prove it.

### The proof (using the law of cosines)

Consider vectors A, B, and the vector C = A - B (the side connecting their tips).

```
        A
       /
      /  C = A - B
     / angle
    O----------> B
```

The **law of cosines** says:

```
|C|^2 = |A|^2 + |B|^2 - 2|A||B|cos(angle)
```

Now expand |C|^2 using C = A - B.

In [ ]:
# Proof: why A.B = |A||B|cos(angle)

A = [3, 4]
B = [4, 0]

print("PROOF: A.B = |A| * |B| * cos(angle)")
print("=" * 55)
print()
print(f"  A = {A},  B = {B}")
print(f"  C = A - B = [{A[0]-B[0]}, {A[1]-B[1]}]")
print()

C = [A[0] - B[0], A[1] - B[1]]
len_A_sq = A[0]**2 + A[1]**2
len_B_sq = B[0]**2 + B[1]**2
len_C_sq = C[0]**2 + C[1]**2

print("Step 1: Compute |C|^2 by expanding C = A - B")
print()
print(f"  |C|^2 = (a1-b1)^2 + (a2-b2)^2")
print(f"        = ({A[0]}-{B[0]})^2 + ({A[1]}-{B[1]})^2")
print(f"        = ({C[0]})^2 + ({C[1]})^2")
print(f"        = {C[0]**2} + {C[1]**2}")
print(f"        = {len_C_sq}")
print()

print("Step 2: Expand the squares")
print()
print("  (a1-b1)^2 + (a2-b2)^2")
print("  = a1^2 - 2*a1*b1 + b1^2 + a2^2 - 2*a2*b2 + b2^2")
print("  = (a1^2 + a2^2) + (b1^2 + b2^2) - 2*(a1*b1 + a2*b2)")
print("  = |A|^2          + |B|^2          - 2*(A.B)")
print()
print(f"  = {len_A_sq} + {len_B_sq} - 2*({A[0]*B[0] + A[1]*B[1]})")
print(f"  = {len_A_sq + len_B_sq} - {2 * (A[0]*B[0] + A[1]*B[1])}")
print(f"  = {len_C_sq}  (matches!)")

In [ ]:
print("Step 3: Compare with the law of cosines")
print()
print("  Law of cosines says:")
print("    |C|^2 = |A|^2 + |B|^2 - 2|A||B|cos(angle)")
print()
print("  We just showed:")
print("    |C|^2 = |A|^2 + |B|^2 - 2*(A.B)")
print()
print("  These must be equal, so:")
print("    2|A||B|cos(angle) = 2*(A.B)")
print()
print("    +-----------------------------------------+")
print("    |  A.B = |A| * |B| * cos(angle)           |")
print("    +-----------------------------------------+")
print()
print("  Rearrange:")
print("    cos(angle) = A.B / (|A| * |B|)")
print()
print("  That's the cosine similarity formula!")
print()

# Verify with numbers
dot_AB = A[0]*B[0] + A[1]*B[1]
len_A = math.sqrt(len_A_sq)
len_B = math.sqrt(len_B_sq)
cos_angle = dot_AB / (len_A * len_B)
angle_rad = math.acos(cos_angle)
angle_deg = math.degrees(angle_rad)

print("Verify with our numbers:")
print(f"  A.B = {A[0]}*{B[0]} + {A[1]}*{B[1]} = {dot_AB}")
print(f"  |A| = sqrt({len_A_sq}) = {len_A:.2f}")
print(f"  |B| = sqrt({len_B_sq}) = {len_B:.2f}")
print(f"  cos(angle) = {dot_AB} / ({len_A:.2f} * {len_B:.2f}) = {cos_angle:.4f}")
print(f"  angle = {angle_deg:.1f} degrees")
print()
print(f"  Check: |A|*|B|*cos(angle) = {len_A:.2f}*{len_B:.2f}*{cos_angle:.4f} = {len_A*len_B*cos_angle:.1f}")
print(f"  A.B = {dot_AB}")
print(f"  They match!")

---
## Part 5: Back to Projection — Connecting Everything

Now we can see the full chain:

```
1. Triangle definition:  cos(angle) = adjacent / hypotenuse

2. With vectors:         adjacent = projection of A onto B
                         hypotenuse = |A|
                         cos(angle) = projection / |A|

3. Projection formula:   projection = A.B / |B|
                         (dot product divided by length of target)

4. Substitute:           cos(angle) = (A.B / |B|) / |A|
                                    = A.B / (|A| * |B|)
```

The right triangle IS there — **projection creates it**.
The dot product **computes the projection** automatically.

In [ ]:
# The full chain with numbers

A = [3, 4]
B = [4, 0]

dot_AB = A[0]*B[0] + A[1]*B[1]  # 12
len_A = math.sqrt(A[0]**2 + A[1]**2)  # 5
len_B = math.sqrt(B[0]**2 + B[1]**2)  # 4

projection = dot_AB / len_B  # adjacent side of the right triangle
cos_from_triangle = projection / len_A  # adjacent / hypotenuse
cos_from_formula = dot_AB / (len_A * len_B)  # dot product formula

print("The Full Chain: Triangle -> Projection -> Dot Product")
print("=" * 55)
print()
print(f"  A = {A}   |A| = {len_A}")
print(f"  B = {B}   |B| = {len_B}")
print(f"  A.B = {dot_AB}")
print()
print("  Step 1: Projection (creates the right angle)")
print(f"    projection of A onto B = A.B / |B|")
print(f"                           = {dot_AB} / {len_B}")
print(f"                           = {projection}")
print()
print("  Step 2: Triangle definition")
print(f"    cos(angle) = adjacent / hypotenuse")
print(f"               = projection / |A|")
print(f"               = {projection} / {len_A}")
print(f"               = {cos_from_triangle}")
print()
print("  Step 3: Same as dot product formula")
print(f"    cos(angle) = A.B / (|A| * |B|)")
print(f"               = {dot_AB} / ({len_A} * {len_B})")
print(f"               = {cos_from_formula}")
print()
print(f"  Triangle result: {cos_from_triangle}")
print(f"  Formula result:  {cos_from_formula}")
print(f"  Identical!")
print()
print("  The dot product formula IS the triangle definition.")
print("  It just computes the projection for you internally.")

In [ ]:
# Why does A.B / |B| give the projection length?
#
# Intuition: the unit vector of B is B_hat = B / |B|
# Projection of A onto B = A . B_hat = A . (B / |B|) = A.B / |B|

print("WHY does A.B / |B| give the projection?")
print("=" * 55)
print()
print("  The unit vector (length=1) in B's direction:")
print(f"    B_hat = B / |B| = {B} / {len_B} = [{B[0]/len_B}, {B[1]/len_B}]")
print()
print("  Projection = how far A goes in B's direction")
print("             = A . B_hat")
print(f"             = {A} . [{B[0]/len_B}, {B[1]/len_B}]")
print(f"             = {A[0]}*{B[0]/len_B} + {A[1]}*{B[1]/len_B}")
print(f"             = {A[0]*B[0]/len_B} + {A[1]*B[1]/len_B}")
print(f"             = {A[0]*B[0]/len_B + A[1]*B[1]/len_B}")
print()
print("  Which is the same as:")
print(f"    A.B / |B| = {dot_AB} / {len_B} = {dot_AB / len_B}")
print()
print("  The dot product with a UNIT vector gives the projection.")
print("  That's the geometric meaning of dot product:")
print("  'how much does A go in the direction of B?'")

---
## Part 6: Try Different Angles

Let's verify the formula works for various angles.

In [ ]:
def cosine_similarity(a, b):
    dot_val = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot_val / (mag_a * mag_b)


print("Verifying: formula matches actual angle")
print("=" * 60)
print()

# Create vectors at known angles from [1, 0]
base = [1, 0]
print(f"  Base vector: {base}")
print()

print(f"  {'Test vector':>16}  {'cos(formula)':>13}  {'cos(actual)':>12}  {'Angle':>6}  Match")
print(f"  {'-'*16}  {'-'*13}  {'-'*12}  {'-'*6}  -----")

test_angles = [0, 30, 45, 60, 90, 120, 180]
for angle_deg in test_angles:
    angle_rad = math.radians(angle_deg)
    test_vec = [round(math.cos(angle_rad), 4),
                round(math.sin(angle_rad), 4)]
    
    cos_formula = cosine_similarity(base, test_vec)
    cos_actual = math.cos(angle_rad)
    match = "yes" if abs(cos_formula - cos_actual) < 0.001 else "NO"
    
    print(f"  {str(test_vec):>16}  {cos_formula:>+13.4f}  {cos_actual:>+12.4f}  {angle_deg:>5}d  {match}")

print()
print("  Every angle matches. The dot product formula and the")
print("  triangle definition always give the same result.")

---
## Part 7: Higher Dimensions — It Still Works

In 2D, you can visualize the angle. In 768 dimensions (like real embeddings),
you can't draw it — but the math is identical.

```
2D:   cos = (a1*b1 + a2*b2) / (|A| * |B|)
768D: cos = (a1*b1 + a2*b2 + ... + a768*b768) / (|A| * |B|)
```

Same formula, more terms. The "angle" exists in the same way,
even though you can't picture a 768-dimensional right triangle.

In [ ]:
import random
random.seed(42)

print("Cosine similarity in high dimensions")
print("=" * 55)
print()

for dims in [2, 10, 100, 768]:
    # Two random vectors
    a = [random.gauss(0, 1) for _ in range(dims)]
    b = [random.gauss(0, 1) for _ in range(dims)]
    
    cos_val = cosine_similarity(a, b)
    angle = math.degrees(math.acos(max(-1, min(1, cos_val))))
    
    print(f"  {dims:>4} dimensions:  cosine = {cos_val:+.4f}  angle = {angle:.1f}deg")

print()
print("  Random vectors in high dimensions tend toward 90 degrees")
print("  (cosine near 0). This is expected — random things are unrelated.")
print()
print("  When an embedding model trains, it pushes similar items")
print("  AWAY from 90 degrees (toward 0 degrees = cosine 1.0).")
print("  That's how 'apple fruit' and 'orange fruit' end up close.")

---
## Part 8: The Visual Summary

```
SCHOOL:                          LINEAR ALGEBRA:

        /|                            A
       / |                           /
 hyp  /  |                          /
     / a |                         /  angle
    /____|         O-----------P-----
    adj                proj    |    B direction
                               |
cos = adj / hyp           projection creates
                          the right angle

THEY'RE THE SAME TRIANGLE.

The dot product formula A.B / (|A| * |B|)
computes the projection internally,
which creates the right angle,
which gives you cos(angle).
```

In [ ]:
# Final demonstration: three equivalent computations

A = [3, 4]
B = [4, 0]

# Method 1: Find actual angle, use math.cos
dot_val = A[0]*B[0] + A[1]*B[1]
len_A = math.sqrt(A[0]**2 + A[1]**2)
len_B = math.sqrt(B[0]**2 + B[1]**2)
cos_formula = dot_val / (len_A * len_B)
angle = math.acos(cos_formula)

# Method 2: Triangle definition (adjacent / hypotenuse)
projection = dot_val / len_B  # adjacent
cos_triangle = projection / len_A  # adjacent / hypotenuse

# Method 3: Direct from math.cos
cos_direct = math.cos(angle)

print("Three ways to compute the same cosine:")
print("=" * 55)
print()
print(f"  A = {A}, B = {B}")
print(f"  Angle between them: {math.degrees(angle):.1f} degrees")
print()
print(f"  Method 1: Dot product formula")
print(f"    A.B / (|A|*|B|) = {dot_val}/({len_A}*{len_B}) = {cos_formula:.4f}")
print()
print(f"  Method 2: Triangle definition")
print(f"    projection = A.B/|B| = {dot_val}/{len_B} = {projection}")
print(f"    cos = adjacent/hyp = {projection}/{len_A} = {cos_triangle:.4f}")
print()
print(f"  Method 3: math.cos(angle)")
print(f"    cos({math.degrees(angle):.1f}deg) = {cos_direct:.4f}")
print()
print(f"  All three: {cos_formula:.4f} = {cos_triangle:.4f} = {cos_direct:.4f}")
print()
print("  The dot product formula does NOT skip the right triangle.")
print("  It BUILDS the right triangle via projection, then computes")
print("  adjacent / hypotenuse. All in one formula.")

---
## Summary

| Question | Answer |
|----------|--------|
| Does cosine need a 90-degree angle? | **Yes, always** |
| Where is the 90-degree angle in the dot product formula? | **Projection creates it** |
| Why does A.B compute projection? | **A.B = |A| * |B| * cos(angle)** (law of cosines proof) |
| What does A.B / |B| mean? | **How far A goes in B's direction** (projection length) |
| Why divide by |A| * |B|? | **Removes length, keeps only direction** |

The chain:
```
dot product  -->  projection  -->  right triangle  -->  cos(angle)
  A.B             A.B / |B|        adj / hyp           direction similarity
```

Your intuition was correct: a right angle IS required.
The dot product formula just creates it for you.